# Chapter 12 — Deployment

Aegis works. It triages, remembers, plans, delegates, routes, and defends itself.

None of that is production. Production is the part where a change nobody fully understood
reaches five hundred alerts a day, and something has to stop it if it is wrong — before a
human notices, and before the bill arrives.

| Control | Stops |
|---|---|
| Evaluation gate | a release that scores worse than what it replaces |
| Canary | a bad release reaching everyone |
| SLOs | silent degradation nobody is watching for |
| Model routing | paying flagship prices for routine work |

**Covered:** §12.2.3 the eval gate · §12.3 canary and rollback · §12.4.2 ceilings and
floors · §12.5.2 model routing · §12.5.3 dated prices and retirements · §12.6 the release
policy — including the gate that blocks a safety improvement.


## Setup

This lab installs from **one** `requirements.txt` file.


In [ ]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


In [ ]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt

Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon. It also confirms this chapter's source folder is in the checkout.


In [ ]:
!python tools/check_env.py --chapter 12

### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [ ]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


## The evaluation gate: arithmetic, not judgment

Chapter 10 gave you numbers. This turns them into a door. It does not get tired, it does
not get talked into anything, and it does not care whose sprint is at stake.

Watch which release gets blocked.


In [ ]:
import sys
sys.path.insert(0, "labs/chapter-12-deployment")   # this chapter's source lives beside the notebook
from deployment.deploy import eval_gate

THRESHOLDS = {"recall": 0.90, "precision": 0.85}

good = {"recall": 0.94, "precision": 0.91}
bad = {"recall": 0.71, "precision": 0.93}      # BETTER precision, worse recall

for label, candidate in (("good release", good), ("bad release ", bad)):
    verdict = eval_gate(candidate, THRESHOLDS)
    status = "PASS" if verdict["passed"] else "BLOCKED"
    print(f'{label}  {status:8} {verdict.get("failures") or ""}')

print()
print("The blocked release had HIGHER precision than the one that passed.")
print("It also missed a quarter of the real attacks. A dashboard that averages")
print("them into one score would have called it an improvement.")


## Canary and rollback

The gate checks a release against a dataset. A dataset is not production. So the new
version goes to a slice of real traffic first and its error rate is compared against the
version already running.

Failures surface at five percent exposure instead of a hundred. Note the second row:
nobody had to notice — the comparison ran and the release reverted itself.


In [ ]:
from deployment.deploy import canary_decision

for label, baseline, canary in (("healthy ", 0.020, 0.025),
                                ("degraded", 0.020, 0.150),
                                ("improved", 0.020, 0.008)):
    result = canary_decision(baseline, canary)
    decision = result.decision if hasattr(result, "decision") else result
    print(f'{label}  baseline={baseline:.3f} canary={canary:.3f} -> {decision}')

print()
print("A rollback here is a config change in seconds, not an incident in hours.")
print("Worth asking your own team: when did we last roll back, and how long did it take?")


## Latency is a ceiling, quality is a floor

The gate and the canary guard *releases*. SLOs guard what degrades with no release at
all: the model updated underneath you, your documents drifted, the alert mix changed.

Two objectives pointing opposite ways. Get them backwards and your alerts fire on
success.


In [ ]:
from deployment.deploy import check_slos

for label, observed in (("healthy ", {"triage_latency_p95_ms": 2100, "escalation_accuracy": 0.97}),
                        ("slow    ", {"triage_latency_p95_ms": 5200, "escalation_accuracy": 0.97}),
                        ("drifting", {"triage_latency_p95_ms": 2100, "escalation_accuracy": 0.88})):
    result = check_slos(observed)
    state = "ok" if result["healthy"] else "ALERT"
    print(f'{label}  {state:6} {list(result["breaches"]) if result["breaches"] else ""}')

print()
print("Sit with the third row. Latency is fine. Nothing crashed. No exception was")
print("thrown, and every dashboard an SRE normally watches is green.")
print("escalation_accuracy fell to 0.88 - wrong escalate/hold calls on 1 alert in 8.")
print("That is Chapter 10's silent failure arriving in production, and the ONLY")
print("reason anyone finds out is that someone wrote the floor down in advance.")


## Dynamic model routing

AI spend does not scale with provisioned capacity; it scales with **use**. Every triage
is metered, and success — adoption — raises the bill.

So route: routine work to a fast tier, deep investigation to the flagship. This is the
biggest cost lever in the system and it changes nothing a user can see.

These numbers come from the book's own cost model, so this chapter and Appendix G agree
to the cent. A lab that invents its own costs is how a book contradicts itself.


In [ ]:
from deployment.release_policy import (stage_model, stage_cost, incident_cost,
                                       STAGE_TOKENS, FAST_MODEL, STRONG_MODEL)

print(f'{"stage":20} {"model":24} {"routed $":>10} {"all-strong $":>13}')
for stage in STAGE_TOKENS:
    model = stage_model(stage)
    print(f'{stage:20} {model:24} {stage_cost(stage, model):>10.5f} '
          f'{stage_cost(stage, STRONG_MODEL):>13.5f}')

routed = incident_cost("routed")
all_strong = incident_cost("all_strong")
saved = 100 * (all_strong - routed) / all_strong

print()
print(f'per incident:  routed ${routed:.5f}   all-strong ${all_strong:.5f}')
print(f'saved by routing: {saved:.1f}%')
print()
print(f'at 500 alerts/day: ${routed * 500 * 30:.2f}/mo routed vs '
      f'${all_strong * 500 * 30:.2f}/mo all-strong')


## Date every number

Model prices move quarterly and models get retired. A cost plan built on an undated
price is folklore; one built on a retiring model is a scheduled outage.

This is not a disclaimer. It is the lesson.


In [ ]:
from deployment.release_policy import retirement_warnings, PRICES_VERIFIED

print("prices verified:", PRICES_VERIFIED)
print()
for warning in retirement_warnings():
    print(f'  {warning["model"]:24} retires {warning["retires"]}')
    print(f'  {"":24} {warning["note"]}')
print()
print("Check the date before you trust the table - including this one.")


## The release policy, and the uncomfortable part

You have gates for quality. Nothing yet stops a change that passes every quality check
and doubles the bill. So build the fourth gate.

Then look hard at what it blocks.


In [ ]:
from deployment.release_policy import cost_gate

BASELINE_MIX = {"triage": 100, "investigation": 10}

cheaper = cost_gate(BASELINE_MIX, {"triage": 120, "investigation": 5})
pricier = cost_gate(BASELINE_MIX, {"triage": 100, "investigation": 40})

print("cheaper release:", cheaper)
print("pricier release:", pricier)
print()
print("Now ask what kind of change sends 4x more alerts to deep investigation.")
print("A change that INVESTIGATES the borderline cases instead of closing them.")
print("A change that would have caught the marginal attack. A recall improvement.")


### Both gates, read together

Neither gate is broken. The failure is reading them **separately**.


In [ ]:
recall_improvement = {"recall": 0.97, "precision": 0.88}

quality = eval_gate(recall_improvement, THRESHOLDS)
cost = pricier

print("quality gate:", "PASS" if quality["passed"] else "BLOCKED")
print("cost gate:   ", "PASS" if cost["passed"] else f'BLOCKED ({cost["reason"]}, '
      f'+{cost["delta_pct"]}%)')
print()
if quality["passed"] and not cost["passed"]:
    print("A change that catches MORE attacks, blocked because it costs more.")
    print("A cost gate with no quality context will eventually block the change")
    print("that would have caught the breach - with a reasonable-looking justification.")


### The policy is the deliverable

The resolution is not a cleverer threshold. It is a **policy**, written down before the
argument happens: which gates block and which warn, who may override, and what the
override costs.

Below, the policy is encoded — quality blocks, cost warns — and the recall improvement
ships. That is a defensible choice, and it is a *choice*. Yours may differ. What is not
optional is having one.


In [ ]:
from deployment.release_policy import release

outcome = release(
    candidate_metrics=recall_improvement,
    thresholds=THRESHOLDS,
    candidate_mix={"triage": 100, "investigation": 40},
    baseline_mix=BASELINE_MIX,
    canary_error_rate=0.022,
    observed_slos={"triage_latency_p95_ms": 2100, "escalation_accuracy": 0.97},
    eval_gate=eval_gate,
    canary_decision=canary_decision,
    check_slos=check_slos,
)

for name, verdict in outcome["steps"]:
    print(f'  {name:12} {verdict}')
print()
print("released:", outcome["released"])
print()
print("The recall improvement shipped - because the policy says a cost regression")
print("WARNS and a quality regression BLOCKS. Someone had to decide that, write it")
print("down, and be ready to defend it in a review.")


---

## What you built — and what Aegis became

A release pipeline: quality gated on measurement, released to a slice you can revert,
watched by objectives written in advance, and priced by a routing decision that saves
roughly half the bill without a user noticing.

- **Gates are arithmetic, not judgment.**
- **Canary at five percent, not a hundred.**
- **Latency is a ceiling; quality is a floor.**
- **Gates need each other.** A cost gate read alone will block the change that would
  have caught the breach.

Twelve chapters ago Aegis was four components in a loop: a model, a dict of tools, a list
of messages, and a `for` loop with a bound on it. It is now a hardened, evaluated,
multi-agent SOC assistant with a release pipeline that can refuse to ship it.

Nothing along the way required a framework. Every framework you will meet rearranges
these same parts and gives them new names. That is why the book taught the parts.
